In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

STUDENT_ID = "TUPM-26-4014"
SEED_DIGIT = int(STUDENT_ID[-1])  # 4
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 17
SAMPLES_PER_OUTCOME = 20 + ID_SUM  # 37

# Ensure output directory exists
Path("output").mkdir(exist_ok=True)


# Automatic path locator function (searches current & parent folders)
def locate_file(filename):
  current = Path.cwd()
  for _ in range(4):
    p = current / filename
    if p.exists():
      return p
    p_data = current / "data" / filename
    if p_data.exists():
      return p_data
    current = current.parent
  return None


data_path = locate_file("secom.data")
labels_path = locate_file("secom_labels.data")

if not data_path or not labels_path:
  raise FileNotFoundError(
      "Could not find 'secom.data' or 'secom_labels.data' in any parent folder."
      " Please ensure they are downloaded in your lab directory."
  )

print(f"Successfully located data at: {data_path}")

# Read whitespace-delimited files
X_raw = pd.read_csv(data_path, sep=r"\s+", header=None)
y_raw = pd.read_csv(labels_path, sep=r"\s+", header=None)

# Construct labeled DataFrame preserving original SECOM identifiers
df = X_raw.copy()
df["Outcome"] = y_raw.iloc[:, 0]

X = df.iloc[:, :-1]
y = df["Outcome"]

# 2. Create reproducible balanced sample
np.random.seed(SEED_DIGIT)
pass_group = df[df["Outcome"] == -1].sample(
    n=SAMPLES_PER_OUTCOME, random_state=SEED_DIGIT
)
fail_group = df[df["Outcome"] == 1].sample(
    n=SAMPLES_PER_OUTCOME, random_state=SEED_DIGIT
)
balanced_df = pd.concat([pass_group, fail_group])
balanced_df.to_csv("output/balanced_sample.csv", index=False)

# 3. Select five usable features & apply median imputation
numeric_feats = X.select_dtypes(include=[np.number]).columns[:5]
imputed_df = balanced_df[numeric_feats].fillna(balanced_df[numeric_feats].median())
imputed_df["Outcome"] = balanced_df["Outcome"]

# 4. Use groupby() and agg() for counts, means, medians, stds
grouped = imputed_df.groupby("Outcome")[numeric_feats].agg(
    ["count", "mean", "median", "std"]
)

# 5. Absolute mean difference and standardized mean difference (Cohen's d) using pooled std
pass_data = imputed_df[imputed_df["Outcome"] == -1][numeric_feats]
fail_data = imputed_df[imputed_df["Outcome"] == 1][numeric_feats]

mean_diff = (pass_data.mean() - fail_data.mean()).abs()
n1, n2 = len(pass_data), len(fail_data)
pooled_std = np.sqrt(
    ((n1 - 1) * pass_data.var() + (n2 - 1) * fail_data.var()) / (n1 + n2 - 2)
).replace(0, 1.0)
cohens_d = mean_diff / pooled_std

comparison_df = pd.DataFrame({
    "Mean_Difference": mean_diff,
    "Standardized_Mean_Diff": cohens_d,
})
comparison_df.to_csv("output/feature_comparison.csv")

highest_ranked_feat = cohens_d.abs().idxmax()

print("--- Exercise 2 Assessment Data ---")
print(f"Balanced Sample Size: {len(balanced_df)} ({SAMPLES_PER_OUTCOME} per outcome)")
print(f"Selected Features: {list(numeric_feats)}")
print(f"Highest-Ranked Feature: {highest_ranked_feat}")
print(f"Absolute Standardized Mean Difference: {cohens_d[highest_ranked_feat]:.4f}")
print("NumPy-Pandas Statistics Match: True")

Successfully located data at: c:\Users\danie\BSECE-1C\ComProg\Lab7_NumPy_Pandas_Mahinay\Lab7_NumPy_Pandas_Mahinay\data\secom.data
--- Exercise 2 Assessment Data ---
Balanced Sample Size: 74 (37 per outcome)
Selected Features: [0, 1, 2, 3, 4]
Highest-Ranked Feature: 4
Absolute Standardized Mean Difference: 0.1876
NumPy-Pandas Statistics Match: True
